# Remote runner — GPU job queue for the Colab session

This notebook is a **shim**. It carries no experiment configuration: which
model runs, at what sample size, under which conditions — all of that arrives
as jobs written into Drive by `codenames-experiment job-submit`.

Run all cells once per session. The last cell blocks, polling for work; that
running cell is also what keeps the Colab session alive.

The loop exits on its own after `IDLE_SHUTDOWN_MINUTES` with an empty queue,
so an idle A100 stops burning compute units. Re-run the last cell to resume
polling.

## Setup (run cells 1–3 once per session)

In [1]:
# Cell 1 — Clone or update the package code from GitHub.
# REPO_BRANCH is explicit: a plain pull on a stale default-branch checkout
# silently reinstalls older package code than the runner expects.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"
REPO_BRANCH = "probing"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} log --oneline -1

Cloning into '/content/codenames-interpretability'...
remote: Enumerating objects: 1031, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 1031 (delta 133), reused 115 (delta 114), pack-reused 885 (from 1)
Receiving objects: 100% (1031/1031), 600.90 KiB | 9.10 MiB/s, done.
Resolving deltas: 100% (712/712), done.
a2d56f1 (HEAD -> probing, origin/probing) fix(notebooks): put the package on sys.path before the pre-warm import


In [ ]:
# Cell 2 — Install the package with the [lens] extra (the GPU stages need it).
# The [remote] extra is deliberately NOT installed: the runner reaches the job
# tree through the Drive FUSE mount, so it needs no Drive API client and no
# OAuth credentials of its own.
!pip install -q -e "{REPO_DIR}[lens]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 906.4/906.4 MB 180.6 MB/s eta 0:00:01

In [ ]:
# Cell 3 — Mount Drive and report the GPU this session actually got.
# Colab Pro does not guarantee an A100. Jobs submitted with --expect-gpu are
# refused before touching data when the session came up on something else.
from google.colab import drive
drive.mount("/content/drive")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free // 2**20} MiB free of {total // 2**20} MiB")
else:
    print("WARNING: no CUDA device — jobs declaring expect_gpu will be refused.")

## Pre-warm model weights (recommended before the poll loop)

A job that starts against a cold cache races a ~15 GB download **inside its own
timeout**. If the session drops mid-download the job is stranded with nothing
measured — observed twice on Qwen, where the log stopped at `Downloading shards: 0%`
and the heartbeat went stale.

Warming first turns that download into a separate, restartable step that costs the
job nothing. It is idempotent: already-cached weights are skipped, so re-running
this cell after a reconnect is cheap.


In [ ]:
# Cell 3b — Pre-warm the weights this session will need.
# Set to the models you plan to submit. qwen_random shares Qwen's repo, so
# warming "qwen" covers it too.
PREWARM = ["mistral", "qwen"]

# The editable install from Cell 2 does not reach THIS already-running
# kernel, so put the freshly cloned package on sys.path first (same reason
# notebooks 01-08 do it). Without this the import below raises
# ModuleNotFoundError even though pip install succeeded.
import sys
REPO_DIR = "/content/codenames-interpretability"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from codenames.remote.prewarm import prewarm_models
results = prewarm_models(PREWARM)
for key, where in results.items():
    print(f"{key:12s} {where}")


## Poll loop

In [ ]:
# Cell 4 — Start the poll loop. This cell blocks; that is intended.
#
# IDLE_SHUTDOWN_MINUTES: exit after this long with an empty queue, so an idle
# session stops consuming compute units. MAX_SESSION_HOURS bounds the whole
# session below Colab's own limit.
DRIVE_ROOT = "/content/drive/MyDrive/Codenames-Research"
JOBS_DIR = f"{DRIVE_ROOT}/_jobs"
IDLE_SHUTDOWN_MINUTES = 30
MAX_SESSION_HOURS = 11

!codenames-experiment job-runner \
    --jobs-dir "{JOBS_DIR}" \
    --repo-dir "{REPO_DIR}" \
    --log-dir /content/logs \
    --idle-shutdown-minutes {IDLE_SHUTDOWN_MINUTES} \
    --max-session-hours {MAX_SESSION_HOURS}